# 04 - Data Transformation
Join validated trips with dimensions, derive analytical columns, and build gold aggregates using both PySpark and SQL.

In [ ]:
from pyspark.sql.functions import col, to_timestamp, hour, when, round as spark_round, date_format, dayofweek, to_date, sum as spark_sum, count, avg

In [ ]:
trips = spark.read.table("ola_lakehouse.silver.trips")
customers = spark.read.table("ola_lakehouse.silver.customers")
drivers = spark.read.table("ola_lakehouse.silver.drivers")
vehicles = spark.read.table("ola_lakehouse.silver.vehicles")
locations = spark.read.table("ola_lakehouse.silver.locations")

### Join trips with all dimension tables

In [ ]:
pickup_loc = locations.selectExpr("location_id as pickup_location_id", "area_name as pickup_area", "city as trip_city")
drop_loc = locations.selectExpr("location_id as drop_location_id", "area_name as drop_area")

enriched = trips \
    .join(customers.selectExpr("customer_id","name as customer_name","city"), on="customer_id", how="left") \
    .join(drivers.selectExpr("driver_id","name as driver_name","vehicle_id","rating as driver_rating"), on="driver_id", how="left") \
    .join(vehicles.selectExpr("vehicle_id","vehicle_type","make","model"), on="vehicle_id", how="left") \
    .join(pickup_loc, on="pickup_location_id", how="left") \
    .join(drop_loc, on="drop_location_id", how="left")

enriched.show()

### Derived columns — duration, fare per km, time-of-day bucket

In [ ]:
enriched = enriched.withColumn("pickup_ts", to_timestamp("pickup_ts")).withColumn("drop_ts", to_timestamp("drop_ts"))

enriched = enriched.withColumn(
    "trip_duration_minutes",
    when(col("status") == "COMPLETED", (col("drop_ts").cast("long") - col("pickup_ts").cast("long")) / 60)
)

enriched = enriched.withColumn(
    "fare_per_km",
    when((col("status") == "COMPLETED") & (col("distance_km") > 0), spark_round(col("fare_amount") / col("distance_km"), 2))
)

enriched = enriched.withColumn("pickup_hour", hour("pickup_ts"))
enriched = enriched.withColumn(
    "time_of_day",
    when(col("pickup_hour").between(5,11), "Morning")
    .when(col("pickup_hour").between(12,16), "Afternoon")
    .when(col("pickup_hour").between(17,20), "Evening")
    .otherwise("Night")
)
enriched = enriched.withColumn("day_name", date_format("pickup_ts","EEEE"))
enriched = enriched.withColumn("is_weekend", dayofweek("pickup_ts").isin([1,7]))
enriched = enriched.withColumn("trip_date", to_date("pickup_ts"))

enriched.printSchema()

In [ ]:
enriched.write.format("delta").mode("overwrite").option("overwriteSchema","true").partitionBy("trip_date").saveAsTable("ola_lakehouse.silver.trips_enriched")

### Gold aggregate 1 — daily revenue by city (PySpark)

In [ ]:
daily_revenue = enriched.filter(col("status")=="COMPLETED") \
    .groupBy("trip_date","trip_city") \
    .agg(
        spark_sum("fare_amount").alias("total_revenue"),
        count("trip_id").alias("completed_trips"),
        spark_round(avg("distance_km"),2).alias("avg_distance_km")
    ).orderBy("trip_date","trip_city")

daily_revenue.show()

### Gold aggregate 2 — vehicle type performance (SQL)

In [ ]:
enriched.createOrReplaceTempView("trips_enriched_vw")

In [ ]:
%sql
SELECT
    vehicle_type,
    COUNT(trip_id) AS total_trips,
    SUM(fare_amount) AS total_revenue,
    ROUND(AVG(fare_per_km), 2) AS avg_fare_per_km
FROM trips_enriched_vw
WHERE status = 'COMPLETED'
GROUP BY vehicle_type
ORDER BY total_revenue DESC

### Gold aggregate 3 — driver leaderboard (PySpark)

In [ ]:
leaderboard = enriched.filter(col("status")=="COMPLETED") \
    .groupBy("driver_id","driver_name") \
    .agg(
        count("trip_id").alias("completed_trips"),
        spark_sum("fare_amount").alias("total_revenue"),
        spark_round(avg("customer_rating"),2).alias("avg_customer_rating")
    ).orderBy(col("total_revenue").desc())

leaderboard.show()

### Gold aggregate 4 — cancellation rate by city (SQL)

In [ ]:
%sql
SELECT
    trip_city,
    COUNT(trip_id) AS total_trips,
    SUM(CASE WHEN status = 'CANCELLED' THEN 1 ELSE 0 END) AS cancelled_trips,
    ROUND(SUM(CASE WHEN status = 'CANCELLED' THEN 1 ELSE 0 END) / COUNT(trip_id) * 100, 2) AS cancellation_rate_pct
FROM trips_enriched_vw
GROUP BY trip_city
ORDER BY cancellation_rate_pct DESC